<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/03_AI_Engineer/beginner/01_llm_apis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with LLM APIs

Call OpenAI and Anthropic APIs confidently — with streaming, async, error handling, and cost awareness.

**Topics:** OpenAI API, Anthropic Claude API, streaming, token counting, rate limiting, error handling, async calls

## 1. OpenAI API — Chat Completions with Proper Error Handling

In [ ]:
import os
import time
from openai import OpenAI
from openai import RateLimitError, APIConnectionError, APIStatusError
from typing import Optional

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

def chat(
    prompt: str,
    system: str = 'You are a helpful assistant.',
    model: str = 'gpt-4o-mini',
    temperature: float = 0.7,
    max_tokens: int = 1024,
    retries: int = 3,
) -> Optional[str]:
    """Call OpenAI with exponential backoff retry."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content
        except RateLimitError:
            wait = 2 ** attempt
            print(f'Rate limited. Waiting {wait}s...')
            time.sleep(wait)
        except APIConnectionError as e:
            print(f'Connection error: {e}. Retrying...')
        except APIStatusError as e:
            print(f'API error {e.status_code}: {e.message}')
            return None
    return None

# Example call (requires valid API key)
# result = chat('What is the capital of France?')
# print(result)

print('OpenAI client configured with retry logic.')
print('Models: gpt-4o (best), gpt-4o-mini (fast+cheap), gpt-3.5-turbo (legacy)')
print('Tip: Always set max_tokens to avoid runaway costs.')

## 2. Anthropic Claude API — Messages Format

In [ ]:
import anthropic

claude = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY', 'sk-ant-...'))

def ask_claude(
    prompt: str,
    system: str = 'You are a helpful assistant.',
    model: str = 'claude-haiku-4-5-20251001',  # fastest/cheapest
    max_tokens: int = 1024,
) -> str:
    """Call Anthropic Claude API."""
    message = claude.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system,
        messages=[{'role': 'user', 'content': prompt}],
    )
    # Claude returns a list of content blocks; text is in the first block
    return message.content[0].text

# Key difference from OpenAI:
# - system is a top-level param, not a message
# - response is message.content[0].text
# - max_tokens is REQUIRED (no default)

# Model tiers:
models = {
    'claude-haiku-4-5-20251001': 'Fastest, cheapest — great for simple tasks',
    'claude-sonnet-4-6': 'Balanced — best for most production use cases',
    'claude-opus-4-6': 'Most capable — complex reasoning, highest cost',
}
print('Claude model lineup:')
for model, desc in models.items():
    print(f'  {model}: {desc}')

## 3. Streaming Responses for Better UX

In [ ]:
import sys

def stream_openai(prompt: str, model: str = 'gpt-4o-mini') -> str:
    """Stream tokens as they arrive — users see output immediately."""
    full_response = ''
    with client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        stream=True,  # <-- key flag
    ) as stream:
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ''
            print(delta, end='', flush=True)  # print without newline
            full_response += delta
    print()  # newline at end
    return full_response

def stream_claude(prompt: str, model: str = 'claude-haiku-4-5-20251001') -> str:
    """Stream Claude response tokens."""
    full_response = ''
    with claude.messages.stream(
        model=model,
        max_tokens=1024,
        messages=[{'role': 'user', 'content': prompt}],
    ) as stream:
        for text in stream.text_stream:
            print(text, end='', flush=True)
            full_response += text
    print()
    return full_response

# Simulated streaming (no API key needed for demo)
simulated_tokens = ['Streaming ', 'gives ', 'users ', 'instant ', 'feedback ', 
                    'instead ', 'of ', 'waiting ', 'for ', 'the ', 'full ', 'response.']
print('Simulated streaming output:')
for token in simulated_tokens:
    print(token, end='', flush=True)
    time.sleep(0.05)
print()
print('\nWhy streaming matters: Users see first token in ~200ms vs waiting 5-10s for full response.')

## 4. Token Counting and Cost Estimation

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = 'gpt-4o') -> int:
    """Count tokens before sending to avoid surprises."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

# Pricing per 1M tokens (as of early 2025)
PRICING = {
    'gpt-4o':              {'input': 2.50, 'output': 10.00},
    'gpt-4o-mini':         {'input': 0.15, 'output': 0.60},
    'claude-sonnet-4-6':   {'input': 3.00, 'output': 15.00},
    'claude-haiku-4-5-20251001': {'input': 0.25, 'output': 1.25},
}

def estimate_cost(input_tokens: int, output_tokens: int, model: str) -> float:
    """Estimate API call cost in USD."""
    if model not in PRICING:
        return 0.0
    p = PRICING[model]
    return (input_tokens * p['input'] + output_tokens * p['output']) / 1_000_000

# Demo
sample_prompt = "Explain the difference between supervised and unsupervised learning in 3 sentences."
tokens = count_tokens(sample_prompt)
print(f'Prompt: "{sample_prompt[:60]}..."')
print(f'Token count: {tokens}')
print()
print('Cost comparison for 1000 calls (50 input + 200 output tokens):')
for model, _ in PRICING.items():
    cost = estimate_cost(50, 200, model) * 1000
    print(f'  {model:<35}: ${cost:.4f}')

## 5. Async API Calls for Parallel Requests

In [ ]:
import asyncio
from openai import AsyncOpenAI
from typing import list

async_client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

async def async_chat(prompt: str, semaphore: asyncio.Semaphore) -> str:
    """Async chat with concurrency limit via semaphore."""
    async with semaphore:  # limit to N concurrent requests
        response = await async_client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=200,
        )
        return response.choices[0].message.content

async def batch_process(prompts: list[str], max_concurrent: int = 5) -> list[str]:
    """Process many prompts in parallel with rate limit protection."""
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [async_chat(p, semaphore) for p in prompts]
    return await asyncio.gather(*tasks, return_exceptions=True)

# Performance comparison
import time
n_prompts = 10
avg_latency_ms = 800  # typical API latency

sequential_time = n_prompts * avg_latency_ms / 1000
parallel_time = avg_latency_ms / 1000  # all run at same time

print(f'Processing {n_prompts} prompts:')
print(f'  Sequential: ~{sequential_time:.1f}s')
print(f'  Parallel (async): ~{parallel_time:.1f}s')
print(f'  Speedup: {sequential_time/parallel_time:.0f}x faster')
print()
print('Key: asyncio.Semaphore(5) limits to 5 concurrent requests')
print('This respects rate limits while maximizing throughput.')

# Run with: asyncio.run(batch_process(prompts))
# In Jupyter: await batch_process(prompts)

## Exercises

1. **Cost calculator:** Build a function that takes a system prompt, list of user messages, and model name, then estimates the total cost for all calls.

2. **Retry with jitter:** Modify the `chat()` function to add random jitter to the backoff (e.g., `wait = 2**attempt + random.uniform(0, 1)`) and explain why jitter prevents thundering herd.

3. **Model comparison:** Write a function that sends the same prompt to `gpt-4o-mini` and `claude-haiku-4-5-20251001` in parallel using `asyncio.gather()`, then prints both responses side-by-side for comparison.